# 03 — Integração PAM/IBGE + MapBiomas Solo

## Projeto AgroESG — Correção de Escopo

**Cultura:** soja  
**Regiões:** Centro-Oeste e Sul  
**Período:** 2019–2024

Objetivo: integrar produção municipal de soja e carbono orgânico do solo, mantendo somente os municípios do Centro-Oeste e Sul e recalculando a priorização agroambiental dentro desse novo universo analítico.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


In [ ]:
BASE_DIR = Path.cwd().parent

print("Diretório base:")
print(BASE_DIR)


In [ ]:
PAM_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "ibge_pam"
)

SOLO_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "mapbiomas_solo"
)

print("PAM:")
print(PAM_DIR)

print("\nMapBiomas Solo:")
print(SOLO_DIR)


In [ ]:
print("Arquivos PAM:")
for arquivo in PAM_DIR.glob("*"):
    print("-", arquivo.name)

print("\nArquivos MapBiomas Solo:")
for arquivo in SOLO_DIR.glob("*"):
    print("-", arquivo.name)


## 5. Definição dos arquivos de entrada

As duas entradas `Curated` já foram produzidas no escopo corrigido. O notebook ainda aplica validações explícitas de cultura e região para evitar vazamento de registros fora do projeto.


In [ ]:
arquivo_pam = (
    PAM_DIR
    / "pam_curated_soja_centro_oeste_sul_2019_2024.csv"
)

arquivo_solo = (
    SOLO_DIR
    / "mapbiomas_solo_municipios_centro_oeste_sul_2019_2024_curated.csv"
)

print("PAM:", arquivo_pam)
print("Solo:", arquivo_solo)


## 6. Carregamento das bases CURATED

Os códigos IBGE são carregados como texto para preservar sua utilização como identificadores territoriais e evitar qualquer transformação numérica indevida.


In [ ]:
pam = pd.read_csv(
    arquivo_pam,
    encoding="utf-8-sig",
    dtype={"codigo_ibge": "string"}
)

solo = pd.read_csv(
    arquivo_solo,
    encoding="utf-8-sig",
    dtype={"codigo_ibge": "string"}
)

regioes_projeto = ["Centro-Oeste", "Sul"]

pam = pam[(pam["cultura"] == "soja") & pam["regiao"].isin(regioes_projeto)].copy()
solo = solo[solo["regiao"].isin(regioes_projeto)].copy()

assert set(pam["cultura"].unique()) == {"soja"}
assert set(pam["regiao"].unique()) == set(regioes_projeto)
assert set(solo["regiao"].unique()) == set(regioes_projeto)

print("✅ Entradas carregadas e validadas no novo escopo.")


In [ ]:
pam.head()


In [ ]:
solo.head()


In [ ]:
print("Colunas PAM:")
print(pam.columns.tolist())

print("\nColunas MapBiomas Solo:")
print(solo.columns.tolist())


## 8. Validação dos tipos de dados

Antes da integração, os tipos são verificados para garantir compatibilidade entre as chaves utilizadas no relacionamento.


In [ ]:
print("Tipos PAM:")
print(pam.dtypes)

print("\nTipos MapBiomas Solo:")
print(solo.dtypes)


In [ ]:
print(
    "codigo_ibge PAM:",
    pam["codigo_ibge"].dtype
)

print(
    "codigo_ibge Solo:",
    solo["codigo_ibge"].dtype
)

print(
    "ano PAM:",
    pam["ano"].dtype
)

print(
    "ano Solo:",
    solo["ano"].dtype
)


In [ ]:
print(
    "Anos PAM:",
    sorted(pam["ano"].unique())
)

print(
    "Anos Solo:",
    sorted(solo["ano"].unique())
)


In [ ]:
print(
    "Culturas presentes:",
    pam["cultura"].unique()
)


## 11. Validação das chaves de integração

A granularidade dos datasets é diferente:

- PAM: município × ano × cultura;
- MapBiomas Solo: município × ano.

Por isso, espera-se uma relação muitos-para-um na integração.

Cada registro municipal anual do MapBiomas Solo poderá ser associado a uma linha de soja na PAM.


In [ ]:
duplicados_pam = (
    pam
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano",
            "cultura"
        ]
    )
    .sum()
)

duplicados_solo = (
    solo
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)

print(
    "Duplicados PAM na chave município-ano-cultura:",
    duplicados_pam
)

print(
    "Duplicados Solo na chave município-ano:",
    duplicados_solo
)


In [ ]:
print(
    "Municípios distintos PAM:",
    pam["codigo_ibge"].nunique()
)

print(
    "Municípios distintos Solo:",
    solo["codigo_ibge"].nunique()
)


## 13. Auditoria de correspondência entre as bases

Antes da construção da base integrada definitiva, é realizado um `left join` de auditoria.

O objetivo é verificar se todos os registros agrícolas da PAM encontram um registro correspondente de carbono orgânico do solo para o mesmo município e ano.


In [ ]:
auditoria_merge = pam.merge(
    solo[
        [
            "codigo_ibge",
            "ano",
            "carbono_solo_t_ha",
            "status_dado"
        ]
    ],
    on=[
        "codigo_ibge",
        "ano"
    ],
    how="left",
    indicator=True,
    validate="many_to_one"
)

auditoria_merge["_merge"].value_counts()


In [ ]:
sem_correspondencia = (
    auditoria_merge[
        auditoria_merge["_merge"] != "both"
    ]
)

print(
    "Registros PAM sem correspondência:",
    len(sem_correspondencia)
)

sem_correspondencia[
    [
        "codigo_ibge",
        "municipio",
        "uf",
        "ano",
        "cultura",
        "_merge"
    ]
].head(20)


## 15. Construção da base integrada PAM + MapBiomas Solo

Após a validação das chaves e da cobertura da integração, é construída a base analítica integrada.

Como município, UF e região já existem na PAM, essas colunas não precisam ser novamente importadas do MapBiomas Solo.

São incorporados:

- área territorial municipal;
- carbono orgânico do solo;
- status de disponibilidade do dado.


In [ ]:
base_integrada = pam.merge(
    solo[
        [
            "codigo_ibge",
            "ano",
            "area_km2",
            "carbono_solo_t_ha",
            "status_dado"
        ]
    ],
    on=[
        "codigo_ibge",
        "ano"
    ],
    how="left",
    validate="many_to_one"
)

print(
    "Dimensão da base integrada:",
    base_integrada.shape
)

base_integrada.head(10)


In [ ]:
print(
    "Linhas PAM:",
    len(pam)
)

print(
    "Linhas integradas:",
    len(base_integrada)
)

print(
    "Diferença:",
    len(base_integrada) - len(pam)
)


In [ ]:
base_integrada[
    [
        "carbono_solo_t_ha",
        "status_dado",
        "area_km2"
    ]
].isna().sum()


In [ ]:
base_integrada["status_dado"].value_counts(
    dropna=False
)


## 18. Validação territorial entre PAM e MapBiomas Solo

Embora a integração tenha sido realizada por `codigo_ibge + ano`, é importante verificar se os atributos territoriais associados aos códigos municipais permanecem consistentes entre as bases.

Essa etapa funciona como uma auditoria adicional da integração.


In [ ]:
territorio_solo = (
    solo[
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao"
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "municipio": "municipio_solo",
            "uf": "uf_solo",
            "regiao": "regiao_solo"
        }
    )
)

auditoria_territorial = (
    pam[
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao"
        ]
    ]
    .drop_duplicates()
    .merge(
        territorio_solo,
        on="codigo_ibge",
        how="left",
        validate="one_to_one"
    )
)

auditoria_territorial.head()


In [ ]:
auditoria_territorial["municipio_igual"] = (
    auditoria_territorial["municipio"]
    == auditoria_territorial["municipio_solo"]
)

auditoria_territorial["uf_igual"] = (
    auditoria_territorial["uf"]
    == auditoria_territorial["uf_solo"]
)

auditoria_territorial["regiao_igual"] = (
    auditoria_territorial["regiao"]
    == auditoria_territorial["regiao_solo"]
)

print(
    "Municípios divergentes:",
    (~auditoria_territorial["municipio_igual"]).sum()
)

print(
    "UFs divergentes:",
    (~auditoria_territorial["uf_igual"]).sum()
)

print(
    "Regiões divergentes:",
    (~auditoria_territorial["regiao_igual"]).sum()
)


In [ ]:
auditoria_territorial[
    ~auditoria_territorial["municipio_igual"]
][
    [
        "codigo_ibge",
        "municipio",
        "municipio_solo",
        "uf",
        "uf_solo",
        "regiao",
        "regiao_solo"
    ]
]


In [ ]:
auditoria_territorial[
    ~auditoria_territorial["municipio_igual"]
].to_string(index=False)


### Divergências de nomenclatura municipal

A auditoria territorial compara os nomes de município entre PAM/IBGE e MapBiomas Solo dentro do recorte **Centro-Oeste/Sul**. Eventuais diferenças textuais não impedem a integração quando `codigo_ibge` e `ano` permanecem consistentes, pois essas são as chaves utilizadas no relacionamento.

Nenhuma substituição manual de nome é necessária para realizar o `merge`; a nomenclatura original de cada fonte permanece rastreável.


In [ ]:
assert (
    auditoria_territorial["uf_igual"].all()
), "Existem divergências de UF entre as bases."

assert (
    auditoria_territorial["regiao_igual"].all()
), "Existem divergências de região entre as bases."

print("✅ Divergências encontradas apenas na nomenclatura municipal.")
print("✅ Código IBGE permanece como chave territorial da integração.")


## Validação final da base integrada

Após a auditoria das chaves territoriais, é realizada uma validação final da base integrada PAM × MapBiomas Solo.

A chave analítica esperada é formada por:

`codigo_ibge + ano + cultura`

Cada registro deve representar uma combinação única de município, ano e cultura.


In [ ]:
duplicados_integracao = (
    base_integrada
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano",
            "cultura"
        ]
    )
    .sum()
)

print(
    "Duplicados na chave município-ano-cultura:",
    duplicados_integracao
)

assert duplicados_integracao == 0

print("✅ Chave analítica da base integrada validada.")


## Estrutura analítica da base integrada

Nesta etapa são verificadas as variáveis disponíveis após a integração das informações agrícolas da PAM com os dados de carbono orgânico do solo do MapBiomas Solo.


In [ ]:
print("Dimensão:", base_integrada.shape)

print("\nColunas:")
for coluna in base_integrada.columns:
    print("-", coluna)


## Indicadores agrícolas e ambientais da soja

Todas as análises a seguir consideram exclusivamente a soja produzida nos municípios do Centro-Oeste e Sul.


In [ ]:
resumo_cultura = (
    base_integrada
    .groupby(
        "cultura",
        as_index=False
    )
    .agg(
        municipios=(
            "codigo_ibge",
            "nunique"
        ),

        area_plantada_total_ha=(
            "area_plantada_ha",
            "sum"
        ),

        producao_total_t=(
            "quantidade_produzida_t",
            "sum"
        ),

        rendimento_medio_kg_ha=(
            "rendimento_medio_kg_ha",
            "mean"
        ),

        carbono_solo_medio_t_ha=(
            "carbono_solo_t_ha",
            "mean"
        )
    )
)

resumo_cultura


## Evolução temporal da produção agrícola e do carbono do solo

A análise temporal permite observar conjuntamente a evolução da produção agrícola e o comportamento do estoque médio de carbono orgânico do solo entre 2019 e 2024.

Os resultados representam associações observacionais e não devem ser interpretados como relação causal.


In [ ]:
resumo_ano_cultura = (
    base_integrada
    .groupby(
        [
            "ano",
            "cultura"
        ],
        as_index=False
    )
    .agg(
        area_plantada_total_ha=(
            "area_plantada_ha",
            "sum"
        ),

        producao_total_t=(
            "quantidade_produzida_t",
            "sum"
        ),

        rendimento_medio_kg_ha=(
            "rendimento_medio_kg_ha",
            "mean"
        ),

        carbono_solo_medio_t_ha=(
            "carbono_solo_t_ha",
            "mean"
        )
    )
)

resumo_ano_cultura


In [ ]:
resumo_ano_cultura[
    resumo_ano_cultura["cultura"] == "soja"
]


In [ ]:
# Célula ajustada ao escopo exclusivo de soja.
assert set(base_integrada["cultura"].dropna().unique()) == {"soja"}


## Associação entre indicadores agrícolas e carbono orgânico do solo

A correlação é calculada para a soja dentro do recorte Centro-Oeste/Sul.


In [ ]:
variaveis_correlacao = [
    "area_plantada_ha",
    "area_colhida_ha",
    "quantidade_produzida_t",
    "rendimento_medio_kg_ha",
    "carbono_solo_t_ha"
]

correlacao_soja = (
    base_integrada[
        base_integrada["cultura"] == "soja"
    ][variaveis_correlacao]
    .corr()
)

correlacao_soja


In [ ]:
correlacao_soja = base_integrada[variaveis_correlacao].corr(numeric_only=True)
correlacao_soja


## Interpretação inicial das correlações

As correlações são exploratórias e descrevem apenas associações observadas entre os indicadores municipais de soja e o carbono orgânico do solo no Centro-Oeste e Sul. Não devem ser interpretadas como causalidade.


## Indicadores agroambientais por região

A análise regional permite comparar a produção agrícola e o estoque médio de carbono orgânico do solo entre as regiões Centro-Oeste e Sul.

Essa abordagem reduz parcialmente o efeito da heterogeneidade territorial existente entre diferentes áreas do país.


In [ ]:
resumo_regiao_cultura = (
    base_integrada
    .groupby(
        [
            "regiao",
            "cultura"
        ],
        as_index=False
    )
    .agg(
        municipios=(
            "codigo_ibge",
            "nunique"
        ),

        area_plantada_total_ha=(
            "area_plantada_ha",
            "sum"
        ),

        producao_total_t=(
            "quantidade_produzida_t",
            "sum"
        ),

        rendimento_medio_kg_ha=(
            "rendimento_medio_kg_ha",
            "mean"
        ),

        carbono_solo_medio_t_ha=(
            "carbono_solo_t_ha",
            "mean"
        )
    )
)

resumo_regiao_cultura


In [ ]:
(
    resumo_regiao_cultura[
        resumo_regiao_cultura["cultura"] == "soja"
    ]
    .sort_values(
        "producao_total_t",
        ascending=False
    )
)


In [ ]:
# Célula ajustada ao escopo exclusivo de soja.
assert set(base_integrada["cultura"].dropna().unique()) == {"soja"}


## Principais municípios produtores de soja

O ranking de produção considera apenas os municípios do Centro-Oeste e Sul presentes no escopo corrigido.


In [ ]:
top_soja = (
    base_integrada[
        base_integrada["cultura"] == "soja"
    ]
    .groupby(
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao"
        ],
        as_index=False
    )
    .agg(
        producao_total_t=(
            "quantidade_produzida_t",
            "sum"
        ),

        area_plantada_total_ha=(
            "area_plantada_ha",
            "sum"
        ),

        carbono_solo_medio_t_ha=(
            "carbono_solo_t_ha",
            "mean"
        )
    )
    .sort_values(
        "producao_total_t",
        ascending=False
    )
    .head(20)
)

top_soja


In [ ]:
top_soja_por_regiao = (
    base_integrada
    .groupby(["regiao", "codigo_ibge", "municipio", "uf"], as_index=False)["quantidade_produzida_t"]
    .sum()
    .sort_values(["regiao", "quantidade_produzida_t"], ascending=[True, False])
)

top_soja_por_regiao.groupby("regiao").head(15)


## Priorização agroambiental exploratória

Após a integração dos dados agrícolas da PAM/IBGE com os dados de carbono orgânico do solo do MapBiomas Solo, é possível construir uma etapa de triagem territorial.

O objetivo desta análise não é estimar créditos de carbono nem classificar diretamente municípios como aptos à geração de créditos.

A proposta é identificar municípios que combinam:

- elevada relevância produtiva;
- elevada área agrícola;
- comportamento desfavorável ou vulnerável do estoque de carbono orgânico do solo.

Esses municípios poderão ser priorizados em análises posteriores com outras dimensões ambientais, como clima, uso e cobertura da terra, queimadas e emissões de gases de efeito estufa.

Os indicadores apresentados nesta etapa devem, portanto, ser interpretados como critérios exploratórios de priorização agroambiental.


In [ ]:
priorizacao_base = (
    base_integrada
    .groupby(
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            "cultura"
        ],
        as_index=False
    )
    .agg(
        producao_total_t=(
            "quantidade_produzida_t",
            "sum"
        ),

        area_plantada_total_ha=(
            "area_plantada_ha",
            "sum"
        ),

        area_colhida_total_ha=(
            "area_colhida_ha",
            "sum"
        ),

        carbono_solo_medio_t_ha=(
            "carbono_solo_t_ha",
            "mean"
        )
    )
)

priorizacao_base.head()


In [ ]:
print("Dimensão:", priorizacao_base.shape)

print(
    "Municípios:",
    priorizacao_base["codigo_ibge"].nunique()
)

print(
    "Culturas:",
    priorizacao_base["cultura"].unique()
)

priorizacao_base.head(10)


In [ ]:
carbono_extremos = (
    base_integrada[
        base_integrada["ano"].isin([2019, 2024])
    ]
    .pivot_table(
        index="codigo_ibge",
        columns="ano",
        values="carbono_solo_t_ha",
        aggfunc="first"
    )
    .reset_index()
)

carbono_extremos.columns.name = None

carbono_extremos = carbono_extremos.rename(
    columns={
        2019: "carbono_2019_t_ha",
        2024: "carbono_2024_t_ha"
    }
)

carbono_extremos.head()


In [ ]:
carbono_extremos[
    [
        "carbono_2019_t_ha",
        "carbono_2024_t_ha"
    ]
].isna().sum()


In [ ]:
carbono_extremos[
    carbono_extremos[
        [
            "carbono_2019_t_ha",
            "carbono_2024_t_ha"
        ]
    ].isna().any(axis=1)
].head(20)


In [ ]:
print(
    "Municípios sem carbono em 2019:",
    carbono_extremos["carbono_2019_t_ha"].isna().sum()
)

print(
    "Municípios sem carbono em 2024:",
    carbono_extremos["carbono_2024_t_ha"].isna().sum()
)

print(
    "Municípios sem pelo menos um dos extremos:",
    carbono_extremos[
        [
            "carbono_2019_t_ha",
            "carbono_2024_t_ha"
        ]
    ].isna().any(axis=1).sum()
)


In [ ]:
priorizacao_base = priorizacao_base.merge(
    carbono_extremos,
    on="codigo_ibge",
    how="left",
    validate="many_to_one"
)

print(
    "Dimensão após merge:",
    priorizacao_base.shape
)

priorizacao_base.head()


In [ ]:
print(
    "Carbono 2019 ausente:",
    priorizacao_base[
        "carbono_2019_t_ha"
    ].isna().sum()
)

print(
    "Carbono 2024 ausente:",
    priorizacao_base[
        "carbono_2024_t_ha"
    ].isna().sum()
)


In [ ]:
assert len(priorizacao_base) == priorizacao_base["codigo_ibge"].nunique()
assert set(priorizacao_base["cultura"].unique()) == {"soja"}
assert set(priorizacao_base["regiao"].unique()) == {"Centro-Oeste", "Sul"}

print("✅ Merge dos extremos de carbono concluído sem alteração da cardinalidade.")


In [ ]:
priorizacao_base["variacao_carbono_t_ha"] = (
    priorizacao_base["carbono_2024_t_ha"]
    - priorizacao_base["carbono_2019_t_ha"]
)


In [ ]:
import numpy as np

priorizacao_base["variacao_carbono_pct"] = np.where(
    priorizacao_base["carbono_2019_t_ha"] > 0,

    (
        (
            priorizacao_base["carbono_2024_t_ha"]
            - priorizacao_base["carbono_2019_t_ha"]
        )
        / priorizacao_base["carbono_2019_t_ha"]
        * 100
    ),

    np.nan
)


In [ ]:
priorizacao_base[
    [
        "codigo_ibge",
        "municipio",
        "uf",
        "cultura",
        "carbono_2019_t_ha",
        "carbono_2024_t_ha",
        "variacao_carbono_t_ha",
        "variacao_carbono_pct"
    ]
].head(10)


In [ ]:
priorizacao_base[
    "variacao_carbono_pct"
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)


In [ ]:
# Score Produção (0 a 1)
priorizacao_base["score_producao"] = (
    priorizacao_base
    .groupby("cultura")[
        "producao_total_t"
    ]
    .rank(
        pct=True,
        method="average"
    )
)

# Score Área (0 a 1)
priorizacao_base["score_area"] = (
    priorizacao_base
    .groupby("cultura")[
        "area_plantada_total_ha"
    ]
    .rank(
        pct=True,
        method="average"
    )
)


### Componente de Vulnerabilidade do Carbono

Invertemos a variação para que perdas representem valores positivos de redução:
* **Próximo de 1:** Maior redução absoluta de carbono do solo dentro da respectiva cultura.
* **Próximo de 0:** Menor redução ou aumento do carbono do solo.


In [ ]:
# Inverter sinal para focar na redução
priorizacao_base["reducao_carbono_t_ha"] = (
    -priorizacao_base["variacao_carbono_t_ha"]
)

# Score Redução Carbono (0 a 1) - Agrupado por cultura (CORRIGIDO)
priorizacao_base["score_reducao_carbono"] = (
    priorizacao_base
    .groupby("cultura")[
        "reducao_carbono_t_ha"
    ]
    .rank(
        pct=True,
        method="average"
    )
)


In [ ]:
priorizacao_base["indice_priorizacao_agroambiental"] = (
    priorizacao_base["score_producao"] * 0.40
    +
    priorizacao_base["score_area"] * 0.30
    +
    priorizacao_base["score_reducao_carbono"] * 0.30
)


In [ ]:
priorizacao_base[
    "indice_priorizacao_agroambiental"
] = (
    priorizacao_base[
        "indice_priorizacao_agroambiental"
    ]
    * 100
)


In [ ]:
priorizacao_base[
    [
        "municipio",
        "uf",
        "regiao",
        "cultura",
        "producao_total_t",
        "area_plantada_total_ha",
        "carbono_2019_t_ha",
        "carbono_2024_t_ha",
        "variacao_carbono_t_ha",
        "score_producao",
        "score_area",
        "score_reducao_carbono",
        "indice_priorizacao_agroambiental"
    ]
].head()


In [ ]:
print(
    "Índice calculado com sucesso:",
    priorizacao_base["indice_priorizacao_agroambiental"].notna().sum()
)
print(
    "Índice NÃO calculável (ausência de dados em 2019 ou 2024):",
    priorizacao_base["indice_priorizacao_agroambiental"].isna().sum()
)

print("\n--- Estatísticas dos Scores ---")
display(
    priorizacao_base[
        [
            "score_producao",
            "score_area",
            "score_reducao_carbono",
            "indice_priorizacao_agroambiental"
        ]
    ].describe()
)


### Construção do índice exploratório

O índice combina três dimensões dentro do universo de municípios produtores de soja do Centro-Oeste e Sul:

- **40% — relevância produtiva**;
- **30% — escala agrícola**;
- **30% — redução do carbono orgânico do solo entre 2019 e 2024**.

As variáveis são convertidas em posições percentuais dentro deste novo escopo. Os pesos são heurísticos e o indicador funciona como ferramenta exploratória de triagem territorial, não como metodologia oficial de certificação ou geração de créditos de carbono.


In [ ]:
ranking_soja = (
    priorizacao_base[
        priorizacao_base["cultura"] == "soja"
    ]
    .sort_values(
        "indice_priorizacao_agroambiental",
        ascending=False
    )
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            "producao_total_t",
            "area_plantada_total_ha",
            "carbono_2019_t_ha",
            "carbono_2024_t_ha",
            "variacao_carbono_t_ha",
            "variacao_carbono_pct",
            "indice_priorizacao_agroambiental"
        ]
    ]
)

ranking_soja.head(20)


In [ ]:
ranking_soja_regiao = (
    priorizacao_base
    .sort_values(["regiao", "indice_priorizacao_agroambiental"], ascending=[True, False])
    [["codigo_ibge", "municipio", "uf", "regiao", "producao_total_t", "area_plantada_total_ha", "indice_priorizacao_agroambiental"]]
)

ranking_soja_regiao.groupby("regiao").head(20)


In [ ]:
priorizacao_base["classe_priorizacao"] = (
    priorizacao_base
    .groupby("cultura")[
        "indice_priorizacao_agroambiental"
    ]
    .transform(
        lambda x: pd.qcut(
            x,
            q=3,
            labels=[
                "baixa",
                "media",
                "alta"
            ],
            duplicates="drop"
        )
    )
)


In [ ]:
priorizacao_base[
    "classe_priorizacao"
].value_counts(
    dropna=False
)


In [ ]:
pd.crosstab(
    priorizacao_base["cultura"],
    priorizacao_base["classe_priorizacao"],
    margins=True
)


## Classificação exploratória da prioridade agroambiental

A classificação em baixa, média e alta prioridade é calculada somente para a soja no recorte Centro-Oeste/Sul.


In [ ]:
alta_prioridade = (
    priorizacao_base[
        priorizacao_base[
            "classe_priorizacao"
        ] == "alta"
    ]
    .sort_values(
        [
            "cultura",
            "indice_priorizacao_agroambiental"
        ],
        ascending=[
            True,
            False
        ]
    )
)

alta_prioridade[
    [
        "codigo_ibge",
        "municipio",
        "uf",
        "regiao",
        "cultura",
        "producao_total_t",
        "area_plantada_total_ha",
        "carbono_2019_t_ha",
        "carbono_2024_t_ha",
        "variacao_carbono_t_ha",
        "indice_priorizacao_agroambiental",
        "classe_priorizacao"
    ]
].head(30)


In [ ]:
alta_prioridade[
    alta_prioridade["cultura"] == "soja"
].head(20)


In [ ]:
alta_prioridade.groupby("regiao").agg(
    municipios=("codigo_ibge", "nunique"),
    indice_medio=("indice_priorizacao_agroambiental", "mean"),
    producao_total_t=("producao_total_t", "sum")
).sort_values("indice_medio", ascending=False)


## Distribuição regional da prioridade agroambiental

Após a classificação dos municípios, é analisada a distribuição das classes de prioridade entre as regiões Centro-Oeste e Sul.

Essa análise permite identificar a concentração territorial dos municípios classificados como alta prioridade e comparar os padrões entre Centro-Oeste e Sul.

A distribuição apresentada deriva exclusivamente do índice exploratório construído neste notebook e não representa classificação oficial de risco ou elegibilidade para projetos de carbono.


In [ ]:
distribuicao_regional = (
    priorizacao_base
    .dropna(
        subset=["classe_priorizacao"]
    )
    .groupby(
        [
            "regiao",
            "cultura",
            "classe_priorizacao"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="municipios"
    )
)

distribuicao_regional


In [ ]:
pd.crosstab(
    [
        priorizacao_base["regiao"],
        priorizacao_base["cultura"]
    ],
    priorizacao_base["classe_priorizacao"],
    margins=True
)


In [ ]:
alta_por_regiao = (
    priorizacao_base[
        priorizacao_base["classe_priorizacao"] == "alta"
    ]
    .groupby(
        [
            "regiao",
            "cultura"
        ],
        as_index=False
    )
    .agg(
        municipios_alta_prioridade=(
            "codigo_ibge",
            "nunique"
        ),
        indice_medio=(
            "indice_priorizacao_agroambiental",
            "mean"
        ),
        producao_total_t=(
            "producao_total_t",
            "sum"
        ),
        area_plantada_total_ha=(
            "area_plantada_total_ha",
            "sum"
        )
    )
)

alta_por_regiao.sort_values(
    [
        "cultura",
        "municipios_alta_prioridade"
    ],
    ascending=[
        True,
        False
    ]
)


In [ ]:
priorizacao_base.groupby(
    [
        "cultura",
        "classe_priorizacao"
    ],
    observed=True
)["indice_priorizacao_agroambiental"].agg(
    [
        "count",
        "min",
        "mean",
        "max"
    ]
)


### Interpretação da distribuição regional

A distribuição permite comparar a concentração de municípios prioritários entre **Centro-Oeste** e **Sul**. O índice combina relevância produtiva, escala agrícola e redução observada de carbono no solo e possui caráter exclusivamente exploratório.


## Exportação dos produtos da integração

São exportados três produtos para `databases_curated/integracao_pam_mapbiomas_solo`:

- base integrada de soja, município-ano;
- base municipal de priorização agroambiental;
- ranking de priorização da soja.

Não é gerado arquivo para outra cultura.


In [ ]:
from pathlib import Path

CURATED_INTEGRACAO_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "integracao_pam_mapbiomas_solo"
)

CURATED_INTEGRACAO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Diretório de saída:")
print(CURATED_INTEGRACAO_DIR)


In [ ]:
print("=== OBJETOS QUE SERÃO EXPORTADOS ===")
print("\nBase integrada:", base_integrada.shape)
print("Base de priorização:", priorizacao_base.shape)
print("Ranking soja:", ranking_soja.shape)


In [ ]:
print(
    "\nÍndice calculável:",
    priorizacao_base[
        "indice_priorizacao_agroambiental"
    ].notna().sum()
)

print(
    "Índice não calculável:",
    priorizacao_base[
        "indice_priorizacao_agroambiental"
    ].isna().sum()
)


In [ ]:
ranking_soja_final = (
    priorizacao_base[
        priorizacao_base["cultura"] == "soja"
    ]
    .sort_values(
        "indice_priorizacao_agroambiental",
        ascending=False,
        na_position="last"
    )
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            "cultura",
            "producao_total_t",
            "area_plantada_total_ha",
            "area_colhida_total_ha",
            "carbono_solo_medio_t_ha",
            "carbono_2019_t_ha",
            "carbono_2024_t_ha",
            "variacao_carbono_t_ha",
            "variacao_carbono_pct",
            "score_producao",
            "score_area",
            "score_reducao_carbono",
            "indice_priorizacao_agroambiental",
            "classe_priorizacao"
        ]
    ]
    .reset_index(drop=True)
)


In [ ]:
print("Municípios no ranking final:", len(ranking_soja_final))
print("Regiões:", sorted(ranking_soja_final["regiao"].unique()))


In [ ]:
ranking_soja_final.head()


In [ ]:
ranking_soja_final.head(20)


In [ ]:
arquivo_base_integrada = (
    CURATED_INTEGRACAO_DIR
    / "pam_mapbiomas_solo_integrado_soja_centro_oeste_sul_2019_2024.csv"
)

base_integrada.to_csv(arquivo_base_integrada, index=False, encoding="utf-8-sig")
print("Base integrada salva em:", arquivo_base_integrada)


In [ ]:
arquivo_priorizacao = (
    CURATED_INTEGRACAO_DIR
    / "priorizacao_agroambiental_soja_centro_oeste_sul_2019_2024.csv"
)

priorizacao_base.to_csv(arquivo_priorizacao, index=False, encoding="utf-8-sig")
print("Base de priorização salva em:", arquivo_priorizacao)


In [ ]:
arquivo_ranking_soja = (
    CURATED_INTEGRACAO_DIR
    / "ranking_priorizacao_soja_centro_oeste_sul_2019_2024.csv"
)

ranking_soja_final.to_csv(arquivo_ranking_soja, index=False, encoding="utf-8-sig")
print("Ranking da soja salvo em:", arquivo_ranking_soja)


In [ ]:
print("Nenhum ranking adicional é gerado no escopo corrigido.")


In [ ]:
arquivos_exportados = [
    arquivo_base_integrada,
    arquivo_priorizacao,
    arquivo_ranking_soja
]

for arquivo in arquivos_exportados:
    print(arquivo.name, "→", "OK" if arquivo.exists() else "ERRO")


## Validação pós-exportação

Para garantir a integridade dos produtos gerados, os arquivos exportados são lidos novamente a partir da camada `databases_curated`.

Essa etapa permite verificar se a gravação dos arquivos preservou:

- a quantidade esperada de registros;
- as colunas essenciais;
- a cardinalidade das chaves;
- os valores ausentes;
- as culturas analisadas;
- a ordenação dos rankings;
- a ausência de duplicidades indevidas.

A validação pós-exportação reforça a rastreabilidade e a reprodutibilidade do pipeline.


In [ ]:
base_integrada_validacao = pd.read_csv(arquivo_base_integrada, encoding="utf-8-sig")
priorizacao_validacao = pd.read_csv(arquivo_priorizacao, encoding="utf-8-sig")
ranking_soja_validacao = pd.read_csv(arquivo_ranking_soja, encoding="utf-8-sig")


In [ ]:
print("=== DIMENSÕES APÓS RELEITURA ===")
print("Base integrada:", base_integrada_validacao.shape)
print("Priorização:", priorizacao_validacao.shape)
print("Ranking soja:", ranking_soja_validacao.shape)


In [ ]:
assert len(base_integrada_validacao) == len(base_integrada)
assert len(priorizacao_validacao) == len(priorizacao_base)
assert len(ranking_soja_validacao) == len(ranking_soja_final)
print("✅ Quantidade de registros preservada após exportação.")


In [ ]:
print(
    "Culturas na base integrada:",
    sorted(
        base_integrada_validacao[
            "cultura"
        ].dropna().unique()
    )
)

print(
    "Culturas na priorização:",
    sorted(
        priorizacao_validacao[
            "cultura"
        ].dropna().unique()
    )
)


In [ ]:
duplicados_integrada = (
    base_integrada_validacao
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano",
            "cultura"
        ]
    )
    .sum()
)

print(
    "Duplicados na chave município-ano-cultura:",
    duplicados_integrada
)


In [ ]:
duplicados_priorizacao = (
    priorizacao_validacao
    .duplicated(
        subset=[
            "codigo_ibge",
            "cultura"
        ]
    )
    .sum()
)

print(
    "Duplicados na chave município-cultura:",
    duplicados_priorizacao
)


In [ ]:
print(
    "Índice calculável após releitura:",
    priorizacao_validacao[
        "indice_priorizacao_agroambiental"
    ].notna().sum()
)

print(
    "Índice não calculável após releitura:",
    priorizacao_validacao[
        "indice_priorizacao_agroambiental"
    ].isna().sum()
)


In [ ]:
priorizacao_validacao[
    "classe_priorizacao"
].value_counts(
    dropna=False
)


In [ ]:
assert ranking_soja_validacao["indice_priorizacao_agroambiental"].dropna().is_monotonic_decreasing
print("✅ Ranking preserva a ordenação decrescente do índice.")


In [ ]:
assert set(ranking_soja_validacao["cultura"].dropna().unique()) == {"soja"}
assert set(ranking_soja_validacao["regiao"].dropna().unique()) == {"Centro-Oeste", "Sul"}
print("✅ Ranking contém somente soja e as regiões do projeto.")


In [ ]:
assert duplicados_integrada == 0
assert duplicados_priorizacao == 0
assert set(base_integrada_validacao["cultura"].dropna().unique()) == {"soja"}
assert set(base_integrada_validacao["regiao"].dropna().unique()) == {"Centro-Oeste", "Sul"}
assert set(priorizacao_validacao["cultura"].dropna().unique()) == {"soja"}
assert set(priorizacao_validacao["regiao"].dropna().unique()) == {"Centro-Oeste", "Sul"}

print("✅ Validação pós-exportação concluída com sucesso.")


## Conclusão

O notebook integra a PAM/IBGE e o MapBiomas Solo para **soja**, no período 2019–2024, restrito aos municípios das regiões **Centro-Oeste** e **Sul**.

A priorização agroambiental é recalculada dentro desse recorte, garantindo que os percentis de produção, área e redução de carbono sejam coerentes com o novo universo analítico. Ao final são exportadas a base integrada, a base de priorização e o ranking de soja.
